# S&P 500 Constituents

Scrapes the S&P 500 Index Components from [slickcharts.com/sp500](https://www.slickcharts.com/sp500) and saves the result to `sp500_constituents/`.

**Output:** `symbol` ordered by market-cap weight descending.  
Row 0 = rank 1 (largest market cap).

In [8]:
import re
from datetime import date
from io import StringIO
from pathlib import Path

import pandas as pd
import requests

OUTPUT_DIR = Path("sp500_constituents")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
URL = "https://www.slickcharts.com/sp500"

# slickcharts blocks the default Python user-agent; spoof a browser header
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(URL, headers=HEADERS, timeout=30)
response.raise_for_status()
print(f"HTTP {response.status_code} — fetched {len(response.content):,} bytes")

HTTP 200 — fetched 369,095 bytes


In [10]:
# pandas.read_html parses every <table>; the first one is the constituents table.
# Wrap in StringIO to avoid the FutureWarning from pandas.
tables = pd.read_html(StringIO(response.text))
print(f"Found {len(tables)} table(s) on the page")

df = tables[0].copy()
print(f"Shape: {df.shape}")
df.head()

Found 3 table(s) on the page
Shape: (503, 7)


,#,Company,Symbol,Weight,Â Â Â Â Â Â Price,Chg,% Chg
0,1,Nvidia,NVDA,7.11%,Â Â 182.48,5.29,(2.99%)
1,2,Apple Inc.,AAPL,6.23%,Â Â 264.72,0.54,(0.20%)
2,3,Microsoft,MSFT,4.75%,Â Â 398.55,5.81,(1.48%)
3,4,Amazon,AMZN,3.59%,Â Â 208.39,-1.61,(-0.77%)
4,5,Alphabet Inc. (Class A),GOOGL,3.07%,Â Â 306.52,-5.24,(-1.68%)


In [11]:
# ---- normalise column names ----
# slickcharts injects non-breaking spaces into the Price header via CSS padding;
# strip all whitespace and non-ASCII junk before renaming.
df.columns = [
    re.sub(r"[^\x00-\x7F]+", "", c).strip()
    for c in df.columns
]
print("Cleaned columns:", df.columns.tolist())

rename_map = {
    "#": "rank",
    "Company": "company",
    "Symbol": "symbol",
    "Weight": "weight_pct",
    "Price": "price",
    "Chg": "change",
    "% Chg": "pct_change",
}
df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)
print("Renamed columns:", df.columns.tolist())

Cleaned columns: ['#', 'Company', 'Symbol', 'Weight', 'Price', 'Chg', '% Chg']
Renamed columns: ['rank', 'company', 'symbol', 'weight_pct', 'price', 'change', 'pct_change']


In [12]:
# ---- keep only symbol, preserve market-cap rank order ----
out = df[["symbol"]].reset_index(drop=True)
print(f"Extracted {len(out)} symbols")
out.head(5)

Extracted 503 symbols


,symbol
0,NVDA
1,AAPL
2,MSFT
3,AMZN
4,GOOGL


In [13]:
# ---- save (monthly cadence: YYYYMM) ----
month_str = date.today().strftime("%Y%m")
csv_path = OUTPUT_DIR / f"sp500_constituents_{month_str}.csv"

out.to_csv(csv_path, index=False)
print(f"Saved {len(out)} rows → {csv_path}")

Saved 503 rows → sp500_constituents/sp500_constituents_202603.csv


In [14]:
# ---- sanity checks ----
print(f"Total symbols : {len(out)}")
print(f"\nTop 10 (rank 1 → 10):")
display(out.head(10))
print(f"\nBottom 3:")
display(out.tail(3))

Total symbols : 503

Top 10 (rank 1 → 10):


,symbol
0,NVDA
1,AAPL
2,MSFT
3,AMZN
4,GOOGL
5,GOOG
6,META
7,TSLA
8,AVGO
9,BRK.B



Bottom 3:


,symbol
500,PAYC
501,LW
502,NWS
